In [1]:
import pandas as pd

data = pd.read_parquet("../thesaurus_data/thesaurus_with_unmatched_final.parquet")
data = data[["term", "contexts", "definition"]]
data = data.dropna()
data = data.sample(100, random_state=42)
data

,term,contexts,definition
84,ALK-позитивная,"[, Пациенту 25 лет с рецидивирующим течением а...","Генетический маркер, указывающий на наличие пе..."
2470,лучевая терапия на шейно-надключичную область,"[Терапия 6 курсов ЕАСОРР, лучевая терапия на ш...","Лучевая терапия, направленная на область шеи и..."
2804,МРТ,"[Перенес удовлетворительно., Продолжена метрон...","Медицинский метод диагностики, основанный на и..."
4987,язвенная болезнь желудка,"[Язвенная болезнь желудка, вне обострения., По...","Хроническое заболевание желудка, характеризующ..."
4924,эндометриоидная киста правого яичника,[Миомэктомия от 26.07.12 г. Эндометриоидная ки...,Эндометриоидная киста правого яичника — это ви...
...,...,...,...
1344,гемокомпоненты,[Получает заместительные трансфузии гемокомпон...,"Компоненты крови, такие как эритроциты, тромбо..."
2856,мукозит ротовой полости 2ст,"[Перелито 3млн CD 34+/кг веса пациента., В ран...",Мукозит ротовой полости 2 стадии — воспалитель...
3380,паравертебральное новообразование,"[35%)., КТ черепа, ОГК, ОБП с КУ 03.02.24.: со...","Новообразование, локализованное в области позв..."
1183,ВИЧ-инфекция 4В,"[Прогрессирование в феврале 2017г,, ножественн...","Стадия ВИЧ-инфекции, соответствующая стадии СП..."


In [ ]:
import yaml
import json
import re
import time
import requests
import pandas as pd
from typing import Optional, Dict, List, Any
from tqdm import tqdm
from collections import Counter

JUDGE_MODELS = [
    "atla/selene-mini:latest",
    "OxW/Vikhr-Nemo-12B-Instruct-R-21-09-24:q6_K",
    "gemma3:12b"           
]
                        

def call_llm(system_prompt: str, user_prompt: str, temperature=0.0, is_json=False, model_name="qwen2.5:14b"):
    """Вызов одной модели Ollama"""
    payload = {
        "model": model_name,
        "prompt": user_prompt,
        "system": system_prompt,
        "stream": False,
        "keep_alive": 0,
        "options": {
            "temperature": temperature,
            "seed": 42,
            "num_ctx": 6144,
            "repetition_penalty": 1.05,
            "num_predict": 1024,
        },
    }
    if is_json:
        payload["format"] = "json"
    r = requests.post("http://10.0.0.9:11434/api/generate", json=payload, timeout=120)
    return r.json()["response"]

def call_llm_ensemble(system_prompt: str, user_prompt: str, models: List[str], temperature=0.0) -> List[Dict]:
    """Опрашивает несколько моделей-судей и возвращает список их распарсенных ответов"""
    results = []
    for model in models:
        raw = call_llm(system_prompt, user_prompt, temperature=temperature, model_name=model)
        parsed = parse_judge_response(raw)
        parsed["_model"] = model
        parsed["_raw"] = raw[:500]
        results.append(parsed)
        time.sleep(0.5)
    return results

def aggregate_judgments(judgments: List[Dict]) -> Dict:
    """
    Агрегирует ответы нескольких судей в итоговый вердикт и средние оценки.
    judgments: список распарсенных словарей от каждой модели
    """
    if not judgments:
        return {"final_verdict": "ERROR", "reasoning": "Нет ответов от моделей"}

    verdicts = [j.get("final_verdict", "ERROR") for j in judgments]
    
    
    if "FAIL" in verdicts:
        final_verdict = "FAIL"
    elif verdicts.count("REVIEW") >= 2:
        final_verdict = "REVIEW"
    else:
        final_verdict = "PASS"

    all_scores = []
    for j in judgments:
        scores = j.get("definition_validation", {}).get("scores", {})
        if scores:
            all_scores.append(scores)
    if all_scores:
        avg_scores = {}
        for key in all_scores[0].keys():
            values = [s.get(key, 0.0) for s in all_scores if key in s]
            avg_scores[key] = sum(values) / len(values) if values else 0.0
    else:
        avg_scores = {}

    all_issues = []
    for j in judgments:
        issues = j.get("definition_validation", {}).get("issues", [])
        all_issues.extend(issues)
    unique_issues = list(set(all_issues))

    reasoning = f"Итог по {len(judgments)} судьям: вердикты {verdicts} → {final_verdict}"
    
    return {
        "final_verdict": final_verdict,
        "avg_scores": avg_scores,
        "issues": unique_issues,
        "individual_judgments": judgments,
        "reasoning": reasoning
    }


def build_judge_prompt(term: str, contexts: str, definition: str) -> tuple[str, str]:
    system_prompt = """Ты — консервативный медицинский аудитор. Твоя задача: оценить извлеченный медицинский термин по качеству определения.

КРИТИЧЕСКОЕ ПРАВИЛО АНТИ-ГАЛЛЮЦИНАЦИЙ:
Тебе СТРОГО ЗАПРЕЩЕНО выдумывать расшифровки аббревиатур. Если генератор предложил расшифровку (например, ЖЭС = желудочковая экстрасистолия) и она логически соответствует переданному контексту, ты ОБЯЗАН принять ее как верную. Снижай оценку (accuracy) только в том случае, если определение прямо противоречит смыслу текста или общеизвестным медицинским фактам.

КАЧЕСТВО ОПРЕДЕЛЕНИЯ (0.0 - плохо, 0.5 - частично, 1.0 - отлично):
Точность (accuracy): Нет ли фактических ошибок? Не противоречит ли контексту? (Если термин описательный/нестандартный, но передан верно — это 1.0).
Полнота (completeness): Есть ли родовое понятие? Расшифрованы ли аббревиатуры?
Релевантность (relevance): Соответствует ли смысл определения исходному тексту? Убраны ли личные данные пациента?
Ясность (clarity): Написано ли в академическом стиле?

ФОРМАТ ОТВЕТА (СТРОГО JSON):
{
  "reasoning": "Шаг 1: проанализируй, как термин используется в контексте. Шаг 2: проверь, соответствует ли определение генератора этому смыслу.",
  "definition_validation": {
    "verdict": "OK" | "WEAK" | "INCORRECT" | "MISSING" | "HALLUCINATION",
    "scores": {
      "accuracy": 0.0 | 0.5 | 1.0,
      "completeness": 0.0 | 0.5 | 1.0, 
      "relevance": 0.0 | 0.5 | 1.0,
      "clarity": 0.0 | 0.5 | 1.0
    },
    "issues": ["список конкретных проблем, только если есть реальные ошибки"] | []
  },
  "final_verdict": "PASS" | "REVIEW" | "FAIL"
}

Правила final_verdict:
- PASS: type_validation = "OK" И accuracy == 1.0 И все остальные метрики >= 0.5.
- REVIEW: type_validation = "MISMATCH" ИЛИ accuracy == 0.5 ИЛИ две метрики < 1.0.
- FAIL: accuracy == 0.0 ИЛИ verdict = "HALLUCINATION".

Если definition равен null, и ты согласен, что термин в данном контексте амбивалентен или не расшифрован — ставь PASS. Это считается корректной работой системы по предотвращению галлюцинаций."""

    user_prompt = f"""Исходный контекст, из которого взят термин: "{contexts}"

    Термин: "{term}"
    Сгенерированное определение: "{definition}"

    Проведи валидацию и верни результат в формате JSON."""
   
    return system_prompt, user_prompt


def parse_judge_response(response: str) -> Dict:
    """Извлекает JSON из ответа с несколькими стратегиями"""
    if not response:
        return {"error": "empty_response", "final_verdict": "ERROR"}
    
    try:
        
        start_idx = response.find('{')
        end_idx = response.rfind('}')
        if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
            json_str = response[start_idx:end_idx+1]
            return json.loads(json_str)
    except json.JSONDecodeError:
        pass
    
    result = {
        "reasoning": "Ошибка парсинга JSON",
        "definition_validation": {"verdict": "UNCLEAR", "scores": {}, "issues": []},
        "final_verdict": "REVIEW"
    }
    
    if '"final_verdict": "PASS"' in response or '"final_verdict":"PASS"' in response:
        result["final_verdict"] = "PASS"
    elif '"final_verdict": "FAIL"' in response or '"final_verdict":"FAIL"' in response:
        result["final_verdict"] = "FAIL"
        
    result["_raw_response"] = response[:500]
    return result

def validate_terms_ensemble(
    df_batch: pd.DataFrame,
    definition_col: str = "definition",
    context_col: str = "contexts",
    models: List[str] = JUDGE_MODELS,
    pause_between: float = 0.2
) -> List[Dict]:
    results = []
    for idx, row in tqdm(df_batch.iterrows(), total=len(df_batch), desc="Валидация ансамблем"):
        term = row["term"]
        definition = row.get(definition_col, "")
        contexts = row.get(context_col, "")
        
        if not definition or pd.isna(definition):
            results.append({
                "index": idx,
                "term": term,
                "definition": "",
                "final_verdict": "FAIL",
                "avg_scores": {},
                "issues": ["Определение отсутствует"],
                "individual_judgments": [],
                "reasoning": "Нет определения"
            })
            continue
        
        sys_prompt, user_prompt = build_judge_prompt(term, contexts, str(definition))
        individual_judgments = call_llm_ensemble(sys_prompt, user_prompt, models)
        aggregated = aggregate_judgments(individual_judgments)
        
        results.append({
            "index": idx,
            "term": term,
            "definition": definition[:200] + "..." if len(definition) > 200 else definition,
            "final_verdict": aggregated["final_verdict"],
            "avg_scores": aggregated["avg_scores"],
            "issues": aggregated["issues"],
            "individual_judgments": aggregated["individual_judgments"],
            "reasoning": aggregated["reasoning"]
        })
        
        time.sleep(pause_between)
    return results

def calculate_validation_metrics_ensemble(results: list) -> dict:
    if not results:
        print("Список результатов пуст.")
        return {}
    
    df = pd.DataFrame(results)
    total_terms = len(df)
    verdicts = df['final_verdict'].value_counts().to_dict()
    
    all_avg_scores = []
    for res in results:
        scores = res.get("avg_scores", {})
        if scores:
            all_avg_scores.append(scores)
    scores_df = pd.DataFrame(all_avg_scores)
    avg_scores = scores_df.mean().to_dict() if not scores_df.empty else {}
    
    all_issues = []
    for res in results:
        all_issues.extend(res.get("issues", []))
    top_issues = dict(Counter(all_issues).most_common(5))
    
    print("="*40)
    print(f"ОТЧЕТ ПО ВАЛИДАЦИИ АНСАМБЛЕМ ({total_terms} терминов)")
    print(f"Модели-судьи: {', '.join(JUDGE_MODELS)}")
    print("="*40)
    print("\n1. ФИНАЛЬНЫЕ ВЕРДИКТЫ:")
    for v_type in ['PASS', 'REVIEW', 'FAIL', 'ERROR']:
        count = verdicts.get(v_type, 0)
        if count:
            print(f"  • {v_type}: {count} ({100*count/total_terms:.1f}%)")
    print("\n2. СРЕДНИЕ ОЦЕНКИ (усреднённые по всем судьям и всем терминам):")
    for metric, score in avg_scores.items():
        print(f"  • {metric.capitalize()}: {score:.2f}")
    print("\n3. ТОП-5 ЧАСТЫХ ПРОБЛЕМ:")
    if top_issues:
        for issue, cnt in top_issues.items():
            print(f"  • [{cnt} раз] {issue}")
    else:
        print("  • Проблем не найдено!")
    print("="*40)
    
    return {
        "total": total_terms,
        "verdicts": verdicts,
        "avg_scores": avg_scores,
        "top_issues": top_issues
    }

In [ ]:
validation_results = validate_terms_ensemble(
    df_batch=data,
    definition_col="definition",
    context_col="contexts",
    models=JUDGE_MODELS,
    pause_between=0.2
)

# Метрики
res = calculate_validation_metrics_ensemble(validation_results)

for res in validation_results:  # первые 5 примеров
    print(f"\nТермин: {res['term']} → итог: {res['final_verdict']}")
    for ind in res['individual_judgments']:
        print(f"  {ind['_model']}: {ind.get('final_verdict')}")

Валидация ансамблем:   0%|          | 0/100 [00:00<?, ?it/s]

Валидация ансамблем: 100%|██████████| 100/100 [28:14<00:00, 16.95s/it]

ОТЧЕТ ПО ВАЛИДАЦИИ АНСАМБЛЕМ (100 терминов)
Модели-судьи: atla/selene-mini:latest, OxW/Vikhr-Nemo-12B-Instruct-R-21-09-24:q6_K, gemma3:12b

1. ФИНАЛЬНЫЕ ВЕРДИКТЫ:
  • PASS: 97 (97.0%)
  • REVIEW: 1 (1.0%)
  • FAIL: 2 (2.0%)

2. СРЕДНИЕ ОЦЕНКИ (усреднённые по всем судьям и всем терминам):
  • Accuracy: 0.98
  • Completeness: 0.98
  • Relevance: 0.99
  • Clarity: 0.97

3. ТОП-5 ЧАСТЫХ ПРОБЛЕМ:
  • [1 раз] Определение не упоминает, что стернальная пункция может быть как диагностической, так и лечебной процедурой, и не всегда связана с заболеваниями крови.
  • [1 раз] Определение не соответствует контексту, в котором термин используется. В контексте речь идет о стернальной пункции в качестве метода диагностики, а не извлечения костного мозга.
  • [1 раз] Определение не отражает контекст иммуногистохимического экспрессирования, что важно в данном случае.
  • [1 раз] Определение не полностью отражает значение термина в контексте кроветворения. Миелоидный росток включает в себя не только клет

In [ ]:
for res in validation_results:  # первые 5 примеров
    print(f"\nТермин: {res['term']} → итог: {res['final_verdict']}")
    for ind in res['individual_judgments']:
        print(f"  {ind['_model']}: {ind.get('final_verdict')}")

In [ ]:
validation_results

('a', 1)